# 01 — Ingest Contoso Data

This notebook downloads the Contoso V2 parquet dataset and writes it as Delta tables into the Fabric Lakehouse (bronze layer).

**Prerequisites:**
- This notebook must be run inside a Fabric Lakehouse notebook environment
- The Lakehouse `contoso_lakehouse` must be attached

**What it does:**
1. Downloads `parquet-100k.7z` from GitHub
2. Extracts the parquet files to `/tmp/contoso/`
3. Reads each parquet file with PySpark
4. Writes Delta tables to the Lakehouse (bronze layer)

**Output tables:** `bronze_sales`, `bronze_customer`, `bronze_product`, `bronze_store`, `bronze_date`, `bronze_currency_exchange`

In [ ]:
# Step 1: Download Contoso parquet-100k dataset
import subprocess

DOWNLOAD_URL = "https://github.com/sql-bi/Contoso-Data-Generator-V2-data/releases/download/ready-to-use-data/parquet-100k.7z"
DOWNLOAD_PATH = "/tmp/contoso-parquet-100k.7z"
EXTRACT_PATH = "/tmp/contoso/"

print(f"Downloading from: {DOWNLOAD_URL}")
subprocess.run(["wget", "-q", "-O", DOWNLOAD_PATH, DOWNLOAD_URL], check=True)
print("Download complete.")

In [ ]:
# Step 2: Extract the 7z archive
import os

os.makedirs(EXTRACT_PATH, exist_ok=True)

# Install 7zip if not available
subprocess.run(["apt-get", "install", "-y", "-q", "p7zip-full"], check=True)
subprocess.run(["7z", "x", DOWNLOAD_PATH, f"-o{EXTRACT_PATH}", "-y"], check=True)

print("Extraction complete. Files:")
for f in os.listdir(EXTRACT_PATH):
    print(f"  {f}")

In [ ]:
# Step 3: Load each parquet file and write as Delta table to Lakehouse

TABLES = [
    "sales",
    "customer",
    "product",
    "store",
    "date",
    "currency_exchange"
]

for table in TABLES:
    parquet_path = os.path.join(EXTRACT_PATH, f"{table}.parquet")
    target_table = f"bronze_{table}"
    
    if not os.path.exists(parquet_path):
        print(f"⚠️  Skipping {table} — file not found: {parquet_path}")
        continue
    
    print(f"Loading {parquet_path} → {target_table}")
    df = spark.read.parquet(parquet_path)
    row_count = df.count()
    
    df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"  ✅ {target_table}: {row_count:,} rows")

print("\nIngestion complete! Bronze tables are ready.")

In [ ]:
# Step 4: Verify — show row counts for all bronze tables
print("Bronze table row counts:")
for table in TABLES:
    target_table = f"bronze_{table}"
    try:
        count = spark.table(target_table).count()
        print(f"  {target_table}: {count:,}")
    except Exception as e:
        print(f"  {target_table}: ⚠️ {e}")

In [ ]:
# Step 5: Inspect continent values for serving layer reference
print("Distinct continent values in bronze_customer:")
spark.table("bronze_customer").select("continent").distinct().orderBy("continent").show()